# Exercises XP: Day 3 — BERT in Practice

Completed notebook: BERT tokenization, sentiment analysis, NER, model comparison, and RAG.

## Exercise 1 — Tokenization with BERT

This example uses a fixed length of 24 tokens. It is longer than the sentence, so padding makes every input in a batch the same length.

In [ ]:
# Run once in a fresh Colab runtime if needed:
# %pip install -q transformers torch

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
sample_sentence = "BERT makes language tasks easier to explore."
print(sample_sentence)

encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,
    return_attention_mask=True,
    return_tensors="pt",
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    marker = " <— special" if token in tokenizer.all_special_tokens else ""
    print(f"{idx:>5} | {token:<12} | {token_id:>5}{marker}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)

### Exercise 1 reflection

`[CLS]` is placed at the start; its final hidden state is commonly used as a sequence-level representation for classification. `[SEP]` marks the end of a sentence (and separates sentences in a pair). The attention mask contains `1` for real tokens and `0` for `[PAD]` positions, preventing self-attention from assigning meaning to padding.

## Exercise 2 — Sentiment analysis pipeline

In [ ]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "The workshop was clear, engaging, and surprisingly useful."
prediction = sentiment_pipeline(sentence)[0]
print(f"Sentence: {sentence}")
print(f"Label: {prediction['label']}; confidence: {prediction['score']:.4f}")
prediction

### Exercise 2 reflection

I expect a `POSITIVE` label because the sentence uses strongly favorable words such as “clear,” “engaging,” and “useful.” The score is the model's confidence for its selected label: a value closer to 1 means the model distinguishes that label more strongly from the alternative, not that the statement is objectively true.

## Exercise 3 — Custom sentiment analyzer

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        clean_text = " ".join(text.strip().split())
        encoded = self.tokenizer(
            clean_text, return_tensors="pt", truncation=True,
            padding=True, max_length=self.max_length
        )
        return {name: tensor.to(self.device) for name, tensor in encoded.items()}

    def predict(self, text: str) -> Dict[str, float]:
        inputs = self.preprocess(text)
        with torch.no_grad():
            logits = self.model(**inputs).logits
            probabilities = torch.softmax(logits, dim=-1)[0]
        label_id = int(torch.argmax(probabilities).item())
        return {
            "label": self.model.config.id2label[label_id],
            "probability": float(probabilities[label_id].item()),
        }

analyzer = BERTSentimentAnalyzer()
samples = [
    "I loved the helpful examples in this lesson.",
    "The application crashed repeatedly and wasted my time.",
    "The book arrived yesterday.",
]
for text in samples:
    print(f"{text}\n  {analyzer.predict(text)}")

## Exercise 4 — BERT for Named Entity Recognition

The recognizer uses token offsets to return original character spans. It skips special tokens and merges adjacent BIO tokens with the same entity type; this also avoids exposing WordPiece fragments (`##...`) as separate entities.

In [ ]:
from transformers import AutoModelForTokenClassification

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def recognize(self, text: str):
        encoded = self.tokenizer(text, return_tensors="pt", return_offsets_mapping=True, truncation=True)
        offsets = encoded.pop("offset_mapping")[0].tolist()
        model_inputs = {name: tensor.to(self.device) for name, tensor in encoded.items()}
        with torch.no_grad():
            prediction_ids = self.model(**model_inputs).logits.argmax(dim=-1)[0].tolist()

        entities = []
        active = None
        for label_id, (start, end) in zip(prediction_ids, offsets):
            if start == end:  # [CLS], [SEP], or padding
                continue
            label = self.model.config.id2label[label_id]
            if label == "O":
                if active:
                    entities.append(active); active = None
                continue
            prefix, entity_type = label.split("-", 1)
            if active and prefix == "I" and active["entity"] == entity_type and start <= active["end"] + 1:
                active["end"] = end
                active["text"] = text[active["start"]:end]
            else:
                if active:
                    entities.append(active)
                active = {"text": text[start:end], "entity": entity_type, "start": start, "end": end}
        if active:
            entities.append(active)
        return entities

ner = BERTNamedEntityRecognizer()
sample_text = "Barack Obama visited Paris and met leaders at the United Nations."
ner.recognize(sample_text)

## Exercise 5 — Comparing BERT and GPT

| Category | BERT | GPT |
|---|---|---|
| Architecture | Encoder-only Transformer; bidirectional self-attention. | Decoder-only Transformer; causal (left-to-right) self-attention. |
| Primary purpose | Learn contextual representations for understanding input text. | Predict the next token to generate coherent continuations. |
| Typical use cases | Classification, NER, extractive QA, semantic search, embeddings. | Chat, drafting, summarization, code/content generation, reasoning. |
| Strengths | Strong contextual understanding; efficient for discriminative tasks. | Flexible zero/few-shot generation and natural language interaction. |
| Weaknesses | Does not naturally generate long text; usually needs task-specific fine-tuning. | Generation can be slower, hallucinate, and be less precise for token-level labeling. |

Both use the Transformer architecture and can be pretrained on large text corpora. The important difference is the training objective: masked-token prediction makes BERT suited to representation learning, whereas next-token prediction makes GPT suited to generation.

## Exercise 6 — BERT inside Retrieval-Augmented Generation

1. In a RAG system, a BERT-style embedding model converts every document chunk into a dense vector. At question time, it encodes the query with the same model, placing semantically related questions and passages near each other in vector space. In practice, a sentence-embedding model derived from BERT is often used rather than the raw `[CLS]` vector.

2. The system stores document vectors together with their text and metadata in a vector database. The database uses similarity search (commonly cosine similarity or dot product, often with an approximate nearest-neighbor index) to return the most relevant chunks for the query vector.

3. The retrieved passages are inserted into the prompt or supplied as context to a generative model such as GPT. GPT then writes an answer grounded in those passages; applications can also show citations back to the retrieved sources. Retrieval happens before generation, so current/private information need not be memorized by GPT.

4. Example: an internal IT help desk can embed product manuals, deployment runbooks, and resolved tickets. For a question such as “How do I rotate the staging API key?”, BERT retrieval finds the relevant runbook section and GPT turns it into a clear, step-by-step answer based on that evidence.